# TED 02 - Limpeza, Saneamento e Análise Exploratória
Projeto: Da Roça Inteligente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../../retail_store_inventory.csv')
df.head()

## 1. Inspeção inicial

In [ ]:
print('Dimensões:', df.shape)
print(df.info())
print('\nAusentes por coluna:')
print(df.isna().sum())
print('Duplicados:', df.duplicated().sum())

## 2. Ajuste de tipos e validações

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
num_cols=['Inventory Level','Units Sold','Units Ordered','Demand Forecast','Price','Discount','Competitor Pricing']
for c in num_cols:
    print(c, 'negativos:', (df[c] < 0).sum())

Foram encontrados 673 valores negativos em `Demand Forecast`. Como demanda representa quantidade esperada e não possui interpretação operacional negativa neste projeto, esses valores são ajustados para zero. Os demais valores extremos são mantidos e analisados, pois podem representar dias de alta demanda.

In [ ]:
negative_forecast = (df['Demand Forecast'] < 0).sum()
df.loc[df['Demand Forecast'] < 0, 'Demand Forecast'] = 0
print('Previsões negativas tratadas:', negative_forecast)

## 3. Estatísticas descritivas

In [ ]:
df[num_cols].describe().T

## 4. Outliers pelo critério IQR

In [ ]:
for c in num_cols:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1
    lo=q1-1.5*iqr; hi=q3+1.5*iqr
    n=((df[c]<lo)|(df[c]>hi)).sum()
    print(c, 'limites:', round(lo,2), round(hi,2), 'possíveis outliers:', n)

## 5. Análise exploratória

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['Units Sold'], bins=30)
plt.title('Distribuição das unidades vendidas')
plt.xlabel('Unidades vendidas'); plt.ylabel('Frequência'); plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df['Units Sold'], df['Demand Forecast'], s=5, alpha=.25)
plt.title('Unidades vendidas x previsão de demanda')
plt.xlabel('Unidades vendidas'); plt.ylabel('Previsão de demanda'); plt.show()

In [ ]:
df.groupby('Category')['Units Sold'].mean().sort_values(ascending=False).plot(kind='bar', figsize=(8,5))
plt.title('Média de unidades vendidas por categoria'); plt.ylabel('Média'); plt.tight_layout(); plt.show()

In [ ]:
cols=['Inventory Level','Units Sold','Units Ordered','Demand Forecast','Price','Discount','Holiday/Promotion','Competitor Pricing']
print(df[cols].corr().round(3))

## 6. Exportação da base tratada

In [ ]:
df.to_csv('../../data/processed/retail_store_inventory_processed.csv', index=False)